# Gold v2 Vision Expert Final — 246 Full × 4 LVLM

최종 Vision Expert ablation입니다.

메인 실험:
- 246개 `gold_v2_final` 이미지 전체
- Vision Expert 4개: PatchCore / Dinomaly / AnomalyCLIP / ReConPatch-lite
- Evidence 5개: score / ROI crop / heatmap / score+ROI / score+heatmap
- LVLM 4개: Qwen / InternVL / MiniCPM / LLaVA
- 총 80개 조건을 모두 평가하며, 50장 smoke winner는 최종 선택 근거로 사용하지 않습니다.
- Luxia는 API timeout/재현성 문제 때문에 최종 비교에서 제외합니다.

Oracle 분석:
- localization GT가 있는 subset에서 GT bbox / GT ROI / GT mask evidence를 별도 upper-bound로 평가
- GT evidence는 deployable 80조건 ranking과 섞지 않습니다.
- GT가 정상/이상 정답을 직접 누설하지 않도록 non-empty defect GT가 있는 이미지의 Q2/Q3만 평가합니다.

모든 중간 결과는 Drive에 저장되며, 재실행 시 완료된 artifact/prediction을 재사용합니다.


In [ ]:
REPO = "arnold6444/lvlm_insplad"
REPO_REF = "main"
RUN_FOLDER = "lvlm_insplad_results/gold_v2_vision_expert_full"
QA_FOLDER = "lvlm_insplad_qa/gold_v2_final"

MODELS = ["qwen3_5_4b","internvl3_5_8b","minicpm_v_4_5","llava_onevision_7b"]
EXPERTS = ["patchcore","dinomaly","anomalyclip","reconpatch_lite"]
CONDITIONS = ["score","roi","heatmap","score_roi","score_heatmap"]

SEED = 42
MAX_TRAIN_PER_ASSET = 200
PATCHCORE_CORESET_RATIO = 0.05
DINOMALY_ENCODER = "vit_small_patch14_reg4_dinov2"
DINOMALY_DECODER_DEPTH = 8
DINOMALY_MAX_STEPS = 300
RECONPATCH_TRAIN_STEPS = 120
RECONPATCH_MAX_TRAIN_PATCHES = 6000
RECONPATCH_MEMORY_RATIO = 0.05
RECONPATCH_BATCH_SIZE = 192
CHECKPOINT_EVERY = 1

UNSUP_ZIP_NAME = "unsupervised_anomaly_detection.zip"
UNSUP_ZIP_SIZE = 1_170_967_719
SUP_ZIP_NAME = "supervised_fault_classification.zip"
SUP_ZIP_SIZE = 874_753_167

EXPECTED_MAIN_RUNS = len(MODELS) * len(EXPERTS) * len(CONDITIONS)  # 4 × 4 × 5 = 80


In [ ]:
import base64, csv, gc, json, os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive")
RUN_ROOT = DRIVE_ROOT / RUN_FOLDER
QA_DIR = DRIVE_ROOT / QA_FOLDER
WORKSPACE = RUN_ROOT / "workspace"
EXPERT_ROOT = RUN_ROOT / "experts_main"
RECON_ROOT = RUN_ROOT / "experts_reconpatch_lite"
MERGED_ROOT = RUN_ROOT / "merged_evidence"
EVAL_DIR = RUN_ROOT / "eval"
ORACLE_DIR = RUN_ROOT / "oracle"
for p in (RUN_ROOT, WORKSPACE, EXPERT_ROOT, RECON_ROOT, MERGED_ROOT, EVAL_DIR, ORACLE_DIR):
    p.mkdir(parents=True, exist_ok=True)

REPO_DIR = Path("/content/lvlm_insplad")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

def run(cmd, *, env=None, check=True):
    cmd = [str(x) for x in cmd]
    print("+", " ".join(cmd[:12]) + (" ..." if len(cmd) > 12 else ""))
    return subprocess.run(cmd, env=env, check=check)

github_token = userdata.get("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("Colab Secret GITHUB_TOKEN이 필요합니다.")
os.environ["GITHUB_TOKEN"] = github_token

try:
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token


auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
git_env = os.environ.copy()
git_env.update({
    "GIT_CONFIG_COUNT":"1",
    "GIT_CONFIG_KEY_0":"http.extraHeader",
    "GIT_CONFIG_VALUE_0":f"AUTHORIZATION: basic {auth}",
    "GIT_TERMINAL_PROMPT":"0",
})
run(["git","clone","--depth","1","--branch",REPO_REF,f"https://github.com/{REPO}.git",REPO_DIR], env=git_env)
del auth, git_env

# Colab CUDA torch는 유지하고 필요한 패키지만 설치.
run([sys.executable,"-m","pip","install","-q",
     "transformers==5.14.0","accelerate==1.14.0","sentencepiece==0.2.2",
     "pandas>=2.2","numpy>=1.26","pillow>=10","tqdm>=4.66",
     "scipy","scikit-learn","timm","open_clip_torch","ftfy","regex",
     "lightning","kornia","anomalib==2.6.0"])
run([sys.executable,"-m","pip","install","-q","-e",f"{REPO_DIR}[dev]"])

print("repo:", subprocess.check_output(["git","rev-parse","HEAD"], cwd=REPO_DIR, text=True).strip())


In [ ]:
# source ZIP + gold QA 확인
def find_unique_sized(filename, expected_size):
    matches = [p for p in DRIVE_ROOT.rglob(filename) if p.is_file() and p.stat().st_size == expected_size]
    if not matches:
        raise FileNotFoundError(f"{filename} ({expected_size} bytes) not found")
    if len(matches) > 1:
        print("동일 크기 후보 여러 개 -> 첫 번째 사용:", matches[:3])
    return matches[0]

SUP_ZIP = find_unique_sized(SUP_ZIP_NAME, SUP_ZIP_SIZE)
UNSUP_ZIP = find_unique_sized(UNSUP_ZIP_NAME, UNSUP_ZIP_SIZE)

master_files = sorted(QA_DIR.glob("master_qa_*.jsonl"))
if not master_files:
    raise FileNotFoundError(f"master_qa_*.jsonl under {QA_DIR}")
MASTER_QA = master_files[-1]

def read_jsonl(path):
    return [json.loads(x) for x in Path(path).read_text(encoding="utf-8").splitlines() if x.strip()]

master_rows = read_jsonl(MASTER_QA)
q1_rows = [r for r in master_rows if r.get("question_type") == "anomaly_detection"]
if len(q1_rows) != 246 or len({str(r["core_id"]) for r in q1_rows}) != 246:
    raise RuntimeError(f"expected 246 Q1/core images, got {len(q1_rows)}")

# Vision Expert query는 exact-hash aligned unsupervised image를 사용.
vision_qa = []
for r in q1_rows:
    vision_qa.append({
        "id": str(r["core_id"]),
        "core_id": str(r["core_id"]),
        "asset_category": str(r["asset_category"]),
        "image": str(r["unsupervised_image"]),
        "answer": str(r["anomaly_gt"]),
    })
VISION_QA = RUN_ROOT / "vision_q1_246.jsonl"
with VISION_QA.open("w", encoding="utf-8") as f:
    for row in vision_qa:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("vision queries:", len(vision_qa))
print("SUP:", SUP_ZIP)
print("UNSUP:", UNSUP_ZIP)


In [ ]:
# unsupervised manifest 자동 탐색
# 기존 builder와 동일 parser로 실제 파싱되는 CSV만 선택합니다.
sys.path.insert(0, str(REPO_DIR / "src"))
from insplad_benchmark.dataset import parse_unsupervised_manifest

expected_assets = {r["asset_category"] for r in vision_qa}
candidate_csvs = []
for pattern in ("*manifest*.csv", "*anomaly*.csv"):
    candidate_csvs.extend(DRIVE_ROOT.rglob(pattern))

seen = set()
ANOMALY_MANIFEST = None
for path in candidate_csvs:
    path = Path(path)
    if path in seen or not path.is_file() or path.stat().st_size > 200_000_000:
        continue
    seen.add(path)
    try:
        rows = parse_unsupervised_manifest(path)
        assets = {str(r["asset_category"]) for r in rows}
        if expected_assets.issubset(assets):
            ANOMALY_MANIFEST = path
            print("manifest selected:", path, "rows=", len(rows))
            break
    except Exception:
        pass

if ANOMALY_MANIFEST is None:
    raise FileNotFoundError(
        "파싱 가능한 unsupervised anomaly manifest CSV를 Drive에서 찾지 못했습니다. "
        "기존 vision-expert 실험에 사용한 manifest CSV가 Drive에 있어야 합니다."
    )


In [ ]:
# 246장 full leakage-safe workspace
run([
    sys.executable, REPO_DIR / "scripts/build_vision_expert_full_data.py",
    "--anomaly-manifest", ANOMALY_MANIFEST,
    "--unsup-zip", UNSUP_ZIP,
    "--qa", VISION_QA,
    "--out-dir", WORKSPACE,
    "--max-train-per-asset", str(MAX_TRAIN_PER_ASSET),
    "--seed", str(SEED),
])

import pandas as pd
workspace_df = pd.read_csv(WORKSPACE / "workspace_manifest.csv")
query_df = workspace_df[workspace_df.role == "query"]
if len(query_df) != 246:
    raise RuntimeError(f"workspace query coverage {len(query_df)} != 246")
print(query_df.groupby("asset_category").size())


In [ ]:
# PatchCore — 완료 asset은 자동 재사용
run([
    sys.executable, REPO_DIR / "scripts/run_patchcore_cohort.py",
    "--workspace-root", WORKSPACE,
    "--out-dir", EXPERT_ROOT,
    "--coreset-ratio", str(PATCHCORE_CORESET_RATIO),
    "--seed", str(SEED),
])


In [ ]:
# Dinomaly — 완료되면 자동 재사용, OOM이면 batch 8→4→2
run([
    sys.executable, REPO_DIR / "scripts/run_dinomaly_cohort.py",
    "--workspace-root", WORKSPACE,
    "--out-dir", EXPERT_ROOT,
    "--encoder", DINOMALY_ENCODER,
    "--decoder-depth", str(DINOMALY_DECODER_DEPTH),
    "--max-steps", str(DINOMALY_MAX_STEPS),
    "--batch-candidates", "8", "4", "2",
    "--seed", str(SEED),
])


In [ ]:
# AnomalyCLIP — pinned official checkpoint, 이미지별 checkpoint 재사용
run([
    sys.executable, REPO_DIR / "scripts/run_anomalyclip_cohort.py",
    "--workspace-root", WORKSPACE,
    "--out-dir", EXPERT_ROOT,
])


In [ ]:
# ReConPatch-lite — 현재 repo의 paper-faithful screening implementation
# 논문 최종 주장 시 이 expert가 우세하면 publication-grade 구현으로 재검증 필요.
run([
    sys.executable, REPO_DIR / "scripts/run_reconpatch_lite_smoke.py",
    "--workspace-root", WORKSPACE,
    "--out-dir", RECON_ROOT,
    "--train-steps", str(RECONPATCH_TRAIN_STEPS),
    "--max-train-patches", str(RECONPATCH_MAX_TRAIN_PATCHES),
    "--memory-ratio", str(RECONPATCH_MEMORY_RATIO),
    "--batch-size", str(RECONPATCH_BATCH_SIZE),
    "--seed", str(SEED),
])


In [ ]:
# 4 expert manifest 통합 + 순수 heatmap overlay 생성
# 기존 overlay에는 yellow ROI box가 함께 있어 heatmap 조건을 confound하므로,
# 최종 실험에서는 box 없는 pure heatmap overlay를 별도로 생성해 overlay_path로 사용합니다.
import numpy as np
from PIL import Image
from insplad_benchmark.vision_expert import normalize_anomaly_map

def read_csv_rows(path):
    with Path(path).open(newline="", encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))

def abs_artifact(root, value):
    p = Path(str(value))
    return p if p.is_absolute() else (Path(root) / p).resolve()

main_manifest = EXPERT_ROOT / "roi_manifest.csv"
recon_manifest = RECON_ROOT / "roi_manifest_reconpatch_lite.csv"
main_rows = read_csv_rows(main_manifest)
recon_rows = read_csv_rows(recon_manifest)

all_rows = []
for rows, root in ((main_rows, EXPERT_ROOT), (recon_rows, RECON_ROOT)):
    for row in rows:
        if row.get("expert") not in EXPERTS:
            continue
        row = dict(row)
        for field in ("anomaly_map_path","roi_path","overlay_path","heatmap_path"):
            if row.get(field):
                row[field] = str(abs_artifact(root, row[field]))
        all_rows.append(row)

index = {(str(r["query_id"]), str(r["expert"])): r for r in all_rows}
if len(index) != 246 * 4:
    raise RuntimeError(f"4-expert manifest coverage: {len(index)} != 984")

pure_root = MERGED_ROOT / "pure_heatmap"
for i, row in enumerate(all_rows, 1):
    source = Path(row["source_image_path"])
    amap = np.load(Path(row["anomaly_map_path"]), allow_pickle=False)
    norm = normalize_anomaly_map(amap)
    heat = np.zeros((*norm.shape, 3), dtype=np.uint8)
    heat[..., 0] = np.rint(255 * norm).astype(np.uint8)
    heat[..., 1] = np.rint(160 * np.square(norm)).astype(np.uint8)
    heat_img = Image.fromarray(heat, mode="RGB")
    with Image.open(source) as h:
        target = h.convert("RGB")
    heat_img = heat_img.resize(target.size, Image.Resampling.BILINEAR)
    overlay = Image.blend(target, heat_img, alpha=0.42)
    out = pure_root / str(row["expert"]) / f"{row['query_id']}__pure_heatmap.png"
    out.parent.mkdir(parents=True, exist_ok=True)
    if not out.is_file():
        overlay.save(out)
    row["overlay_with_roi_box_path"] = row.get("overlay_path","")
    row["overlay_path"] = str(out.resolve())
    row["heatmap_evidence_protocol"] = "pure_heatmap_overlay_without_bbox"

MERGED_MANIFEST = MERGED_ROOT / "roi_manifest_all.csv"
fields = []
for row in all_rows:
    for k in row:
        if k not in fields:
            fields.append(k)
with MERGED_MANIFEST.open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    w.writerows(all_rows)

print("merged:", len(all_rows), MERGED_MANIFEST)


In [ ]:
# 최종 80조건: 4 experts × 5 evidence × 4 LVLM
# prediction JSONL은 조건별 Drive checkpoint라 같은 셀 재실행 시 완료된 응답 재사용.
run([
    sys.executable, REPO_DIR / "scripts/run_gold_v2_vision_expert_full.py",
    "--qa-dir", QA_DIR,
    "--sup-zip", SUP_ZIP,
    "--roi-manifest", MERGED_MANIFEST,
    "--artifact-root", MERGED_ROOT,
    "--out-dir", EVAL_DIR,
    "--models", *MODELS,
    "--experts", *EXPERTS,
    "--conditions", *CONDITIONS,
    "--checkpoint-every", str(CHECKPOINT_EVERY),
])

leaderboard = pd.read_csv(EVAL_DIR / "leaderboard.csv")
coverage = leaderboard[
    leaderboard.model_key.isin(MODELS)
    & leaderboard.expert.isin(EXPERTS)
    & leaderboard.condition.isin(CONDITIONS)
].drop_duplicates(["model_key","expert","condition"])
if len(coverage) != EXPECTED_MAIN_RUNS:
    raise RuntimeError(f"final coverage {len(coverage)} != {EXPECTED_MAIN_RUNS}")
display(pd.read_csv(EVAL_DIR / "screening_summary.csv").head(20))


## Oracle localization evidence

아래 단계는 deployable Vision Expert 80-condition ranking과 별개입니다.

- GT bbox
- GT ROI crop
- GT polygon/mask overlay

를 사용해 “완벽한 위치 정보가 주어졌을 때 LVLM이 얼마나 좋아지는가?”를 봅니다.

정상 이미지에서 empty GT를 주면 anomaly 정답이 노출되므로,
non-empty defect localization GT가 있는 이미지의 `Q2 defect classification + Q3 localization`만 사용합니다.


In [ ]:
# GT bbox / ROI / mask artifact 생성
from PIL import Image, ImageDraw
from zipfile import ZipFile

loc_files = sorted(QA_DIR.glob("localization_qa_*.jsonl"))
if not loc_files:
    raise FileNotFoundError("localization_qa_*.jsonl")
loc_rows_all = read_jsonl(loc_files[-1])

# 같은 core의 localization_gt metadata를 하나로 모음
loc_meta = {}
for row in loc_rows_all:
    gt = row.get("localization_gt") or {}
    if gt and not bool(gt.get("empty_mask", False)) and gt.get("normalized_bboxes_xywh"):
        loc_meta[str(row["core_id"])] = row

print("non-empty localization GT images:", len(loc_meta))

def extract_member(zip_path, member, root):
    out = root / member
    if out.is_file():
        return out
    out.parent.mkdir(parents=True, exist_ok=True)
    with ZipFile(zip_path) as z:
        try:
            data = z.read(member)
        except KeyError:
            norm = member.replace("\\","/").lstrip("/")
            actual = next(n for n in z.namelist() if n.replace("\\","/").lstrip("/") == norm)
            data = z.read(actual)
    out.write_bytes(data)
    return out

oracle_rows = []
for core_id, row in loc_meta.items():
    target_path = extract_member(SUP_ZIP, str(row["image"]), ORACLE_DIR / "_images")
    with Image.open(target_path) as h:
        img = h.convert("RGB")
    W, H = img.size
    gt = row["localization_gt"]
    boxes = gt["normalized_bboxes_xywh"]
    px_boxes = []
    for x,y,w,h in boxes:
        left = int(round(float(x)*W)); top = int(round(float(y)*H))
        right = int(round((float(x)+float(w))*W)); bottom = int(round((float(y)+float(h))*H))
        left=max(0,min(left,W-1)); top=max(0,min(top,H-1))
        right=max(left+1,min(right,W)); bottom=max(top+1,min(bottom,H))
        px_boxes.append((left,top,right,bottom))
    union = (
        min(b[0] for b in px_boxes), min(b[1] for b in px_boxes),
        max(b[2] for b in px_boxes), max(b[3] for b in px_boxes)
    )

    out_dir = ORACLE_DIR / "visual" / core_id
    out_dir.mkdir(parents=True, exist_ok=True)

    bbox_img = img.copy()
    d = ImageDraw.Draw(bbox_img)
    lw = max(3, round(min(W,H)/180))
    for b in px_boxes:
        d.rectangle((b[0],b[1],b[2]-1,b[3]-1), outline=(255,0,0), width=lw)
    bbox_path = out_dir / "gt_bbox.png"
    bbox_img.save(bbox_path)

    roi_path = out_dir / "gt_roi.png"
    img.crop(union).save(roi_path)

    mask = Image.new("L", (W,H), 0)
    md = ImageDraw.Draw(mask)
    polygons = gt.get("normalized_polygons_xy") or []
    if polygons:
        for poly in polygons:
            pts = [(int(round(float(x)*W)), int(round(float(y)*H))) for x,y in poly]
            if len(pts) >= 3:
                md.polygon(pts, fill=180)
    else:
        for b in px_boxes:
            md.rectangle((b[0],b[1],b[2]-1,b[3]-1), fill=180)
    red = Image.new("RGB",(W,H),(255,0,0))
    mask_overlay = Image.composite(red, img, mask)
    mask_overlay = Image.blend(img, mask_overlay, alpha=0.45)
    mask_path = out_dir / "gt_mask_overlay.png"
    mask_overlay.save(mask_path)

    oracle_rows.append({
        "core_id": core_id,
        "target_path": str(target_path.resolve()),
        "gt_bbox_path": str(bbox_path.resolve()),
        "gt_roi_path": str(roi_path.resolve()),
        "gt_mask_path": str(mask_path.resolve()),
        "coarse_region": gt.get("coarse_region",""),
    })

ORACLE_MANIFEST = ORACLE_DIR / "oracle_manifest.csv"
pd.DataFrame(oracle_rows).to_csv(ORACLE_MANIFEST, index=False)
display(pd.DataFrame(oracle_rows).head())


In [ ]:
# Oracle Q2/Q3 × 3 evidence × 4 LVLM
import time, traceback
sys.path.insert(0, str(REPO_DIR / "scripts"))
import run_gold_baselines as base
from insplad_benchmark.hf_adapters import load_hf_adapter, infer_hf

oracle_index = {r["core_id"]: r for r in oracle_rows}
oracle_qa = [
    r for r in loc_rows_all
    if str(r.get("core_id")) in oracle_index
    and r.get("question_type") in {"defect_classification","defect_localization"}
]
oracle_conditions = ["gt_bbox","gt_roi","gt_mask"]

def oracle_prompt(row, condition):
    options = "\n".join(f"{k}. {v}" for k,v in row["options"].items())
    desc = {
        "gt_bbox":"Image 2 shows the ground-truth defect bounding box.",
        "gt_roi":"Image 2 is a crop of the ground-truth defect region.",
        "gt_mask":"Image 2 shows the ground-truth defect mask overlay.",
    }[condition]
    return (
        "You are an industrial visual inspector. Image 1 is the complete TARGET image. "
        + desc +
        " This is ORACLE upper-bound evidence, not a deployable model output. "
        "Answer only the multiple-choice question.\n\n"
        f"Question: {row['question']}\n{options}\n\n"
        "Return exactly one option letter and nothing else."
    )

def oracle_images(row, condition):
    info = oracle_index[str(row["core_id"])]
    key = {"gt_bbox":"gt_bbox_path","gt_roi":"gt_roi_path","gt_mask":"gt_mask_path"}[condition]
    return [Path(info["target_path"]), Path(info[key])]

def pred_path_for(model, condition):
    return ORACLE_DIR / "predictions" / model / f"{condition}.jsonl"

oracle_metrics = []
for model_key in MODELS:
    spec = base.MODEL_REGISTRY[model_key]
    adapter = None
    print("load", model_key)
    adapter = load_hf_adapter(spec)
    try:
        for condition in oracle_conditions:
            pred_path = pred_path_for(model_key, condition)
            existing = {str(r["id"]):r for r in read_jsonl(pred_path)} if pred_path.is_file() else {}
            preds = []
            for i,row in enumerate(oracle_qa,1):
                old = existing.get(str(row["id"]))
                if old and not old.get("error") and str(old.get("output","")).strip():
                    preds.append(old); continue
                images = oracle_images(row, condition)
                prompt_text = oracle_prompt(row, condition)
                started=time.time(); output=""; error=""
                try:
                    output = str(infer_hf(adapter, images, prompt_text))
                except Exception as exc:
                    error=f"{type(exc).__name__}: {exc}"
                choice = base.parse_choice(output, {str(k).upper() for k in row["options"]})
                preds.append({
                    "id":row["id"],"core_id":row["core_id"],"question_type":row["question_type"],
                    "model_key":model_key,"condition":condition,"output":output,
                    "predicted_choice":choice,"latency_s":round(time.time()-started,3),"error":error,
                })
                pred_path.parent.mkdir(parents=True, exist_ok=True)
                base.write_jsonl(pred_path, preds)

            by_id={str(r["id"]):r for r in preds}
            q2=[r for r in oracle_qa if r["question_type"]=="defect_classification"]
            q3=[r for r in oracle_qa if r["question_type"]=="defect_localization"]
            q2_gold=[r["defect_gt"] for r in q2]
            q2_pred=[base.predicted_semantic(r, by_id[str(r["id"])].get("predicted_choice")) for r in q2]
            metric={
                "model_key":model_key,"condition":condition,
                "images":len(oracle_index),"questions":len(oracle_qa),
                "valid_output_rate":sum(by_id[str(r["id"])].get("predicted_choice") is not None for r in oracle_qa)/len(oracle_qa),
                "overall_accuracy":sum(by_id[str(r["id"])].get("predicted_choice")==r["answer"] for r in oracle_qa)/len(oracle_qa),
                "defect_accuracy":sum(by_id[str(r["id"])].get("predicted_choice")==r["answer"] for r in q2)/len(q2) if q2 else None,
                "defect_macro_f1":base.macro_f1(q2_gold,q2_pred) if q2 else None,
                "localization_accuracy":sum(by_id[str(r["id"])].get("predicted_choice")==r["answer"] for r in q3)/len(q3) if q3 else None,
            }
            oracle_metrics.append(metric)
            pd.DataFrame(oracle_metrics).to_csv(ORACLE_DIR/"oracle_leaderboard.csv",index=False)
            print(metric)
    finally:
        adapter=None; gc.collect()
        try:
            import torch
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        except Exception: pass

display(pd.DataFrame(oracle_metrics))


In [ ]:
# 핵심 결과 GitHub publish
publish_files = [
    "leaderboard.csv","metrics_by_asset.csv","run_status.json",
    "screening_summary.csv","run_metadata.json",
]
run([
    sys.executable, REPO_DIR / "scripts/publish_results_to_github.py",
    "--out-dir", EVAL_DIR,
    "--repo", REPO,
    "--branch", REPO_REF,
    "--prefix", "gold_v2_vision_expert_full",
    "--files", *[f for f in publish_files if (EVAL_DIR/f).is_file()],
], env=os.environ.copy())

# Oracle summary도 별도 prefix
oracle_publish = ["oracle_leaderboard.csv","oracle_manifest.csv"]
run([
    sys.executable, REPO_DIR / "scripts/publish_results_to_github.py",
    "--out-dir", ORACLE_DIR,
    "--repo", REPO,
    "--branch", REPO_REF,
    "--prefix", "gold_v2_vision_expert_oracle",
    "--files", *[f for f in oracle_publish if (ORACLE_DIR/f).is_file()],
], env=os.environ.copy())

print("메인 완료:", EVAL_DIR)
print("Oracle 완료:", ORACLE_DIR)


## 완료 후

ChatGPT에 `vision expert full 결과 나왔어`라고 하면 됩니다.

해석 시:
- 50장 smoke는 preliminary
- 246장 80조건이 최종 Vision Expert ablation
- GT bbox/ROI/mask는 4 HF LVLM의 oracle upper-bound로 별도 표기
